In [1]:
import pyspark
from pyspark.sql import SparkSession

In [1]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

25/03/03 23:45:48 WARN Utils: Your hostname, mystuff resolves to a loopback address: 127.0.1.1; using 193.168.147.155 instead (on interface eth0)
25/03/03 23:45:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/03 23:45:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read \
    .option("header", "true") \
    .csv("fhvhv_tripdata_2021-01.csv.gz")
    # .option("InferSchema", True) \

In [3]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [4]:
import pandas as pd

df_pandas = pd.read_csv(
    'fhvhv_tripdata_2021-01_head.csv',
    parse_dates=['pickup_datetime', 'dropoff_datetime']
) 

In [5]:
df_pandas.dtypes

hvfhs_license_num               object
dispatching_base_num            object
pickup_datetime         datetime64[ns]
dropoff_datetime        datetime64[ns]
PULocationID                     int64
DOLocationID                     int64
SR_Flag                        float64
dtype: object

In [6]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [7]:
from pyspark.sql.types import *

In [8]:
schema = StructType([
    StructField('hvfhs_license_num', StringType(), True), 
    StructField('dispatching_base_num', StringType(), True), 
    StructField('pickup_datetime', TimestampType(), True), 
    StructField('dropoff_datetime', TimestampType(), True), 
    StructField('PULocationID', IntegerType(), True), 
    StructField('DOLocationID', IntegerType(), True), 
    StructField('SR_Flag', StringType(), True)]
)

In [9]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("fhvhv_tripdata_2021-01.csv.gz")
    # .option("InferSchema", True) \

In [10]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [11]:
from pyspark import SparkConf

In [12]:
df = df.repartition(24)

In [13]:
# df.write.parquet("fhvhv/2021/01/")

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/home/kantundpeterpan/projects/zoomcamp/zcde_space/week5/fhvhv/2021/01 already exists. Set mode as "overwrite" to overwrite the existing path.

In [14]:
df = spark.read.parquet("fhvhv/2021/01/")

In [15]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [16]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .take(5)

[Row(pickup_datetime=datetime.datetime(2021, 1, 21, 18, 19, 13), dropoff_datetime=datetime.datetime(2021, 1, 21, 19, 14, 27), PULocationID=256, DOLocationID=265),
 Row(pickup_datetime=datetime.datetime(2021, 1, 26, 11, 15, 51), dropoff_datetime=datetime.datetime(2021, 1, 26, 11, 23, 14), PULocationID=165, DOLocationID=165),
 Row(pickup_datetime=datetime.datetime(2021, 1, 19, 19, 11, 47), dropoff_datetime=datetime.datetime(2021, 1, 19, 19, 29, 45), PULocationID=82, DOLocationID=137),
 Row(pickup_datetime=datetime.datetime(2021, 1, 27, 11, 49, 13), dropoff_datetime=datetime.datetime(2021, 1, 27, 11, 53, 15), PULocationID=97, DOLocationID=97),
 Row(pickup_datetime=datetime.datetime(2021, 1, 17, 18, 35, 49), dropoff_datetime=datetime.datetime(2021, 1, 17, 18, 55, 43), PULocationID=74, DOLocationID=116)]

In [17]:
from pyspark.sql import functions as F

In [18]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f"s/{num:03x}"
    elif num % 3 == 0:
        return f"a/{num:03x}"
    else:
        return f"e/{num:03x}"

In [19]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=StringType())

In [20]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/a39| 2021-01-21|  2021-01-21|         256|         265|
|  a/b37| 2021-01-26|  2021-01-26|         165|         165|
|  e/acc| 2021-01-19|  2021-01-19|          82|         137|
|  e/b38| 2021-01-27|  2021-01-27|          97|          97|
|  e/9ce| 2021-01-27|  2021-01-27|         134|          95|
|  s/acd| 2021-01-17|  2021-01-17|          74|         116|
|  e/b38| 2021-01-28|  2021-01-28|          78|          75|
|  a/a7a| 2021-01-04|  2021-01-04|         236|         126|
|  e/9ce| 2021-01-21|  2021-01-21|          79|         137|
|  e/b3c| 2021-01-29|  2021-01-29|         198|         157|
|  e/acc| 2021-01-24|  2021-01-24|          37|          37|
|  e/acc| 2021-01-05|  2021-01-05|         106|          72|
|  e/acc| 2021-01-20|  2021-01-20|          35|         265|
|  e/a39| 2021-01-10|  2

In [2]:
spark = SparkSession.builder \
    .appName('pyspark-run-with-gcp-bucket') \
    .config("spark.jars", "./gcs-connector-hadoop3-latest.jar") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

25/03/04 23:56:58 WARN Utils: Your hostname, mystuff resolves to a loopback address: 127.0.1.1; using 193.168.147.155 instead (on interface eth0)
25/03/04 23:56:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/03/04 23:57:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
# Configure GCS authentication
spark.conf.set("google.cloud.auth.service.account.enable", "true")
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile", 
                                     "../week1/3_intro_terraform/workspaceaddon-436615-4bcf737409b7.json")
spark._jsc.hadoopConfiguration().set('fs.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem')

In [17]:
spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("gs://workspaceaddon-436615/yellow_tripdata_2021-01.csv.gz").schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampType(), True), StructField('tpep_dropoff_datetime', TimestampType(), True), StructField('passenger_count', IntegerType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', IntegerType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', IntegerType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [9]:
spark.read.format("parquet").load("gs://workspaceaddon-436615/yellow_tripdata_2024-01.parquet").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [18]:
spark.stop()